# HS85 End-to-End Workflow (PR #412 style)

This notebook demonstrates one complete flow for adding/verifying a problem in `OptimizationProblems.jl` using **HS85** as a focused test case.

Flow covered:
1. Load extraction artifact (`docs/extract_hs85.json`).
2. Run repository rule checks (files, header, bounds, metadata).
3. Compare extracted bounds with implementation bounds.
4. Instantiate the ADNLP problem and inspect dimensions.
5. Optionally solve the PureJuMP model with Ipopt (if available).

The goal is to make the process reviewable and reproducible in one place.

In [ ]:
using Pkg

repo_root = if isfile(joinpath(pwd(), "Project.toml"))
    pwd()
elseif isfile(joinpath(pwd(), "..", "Project.toml"))
    normpath(joinpath(pwd(), ".."))
else
    error("Could not locate Project.toml from current working directory: $(pwd())")
end

Pkg.activate(repo_root)

using JSON
using OptimizationProblems
using ADNLPModels
using NLPModels

println("Activated project at: ", repo_root)

In [ ]:
extract_path = joinpath(repo_root, "docs", "extract_hs85.json")
extract = JSON.parsefile(extract_path)

println("Loaded extraction file: ", extract_path)
println("Problem: ", extract["problem"])
println("Variables: ", length(extract["variables"]))
println("Constraints listed in JSON: ", length(extract["constraints"]))
println("Uncertainties count: ", length(extract["uncertainties"]))

In [ ]:
function first_lines_have_pattern(path::String, pattern::String; n::Int = 20)
    if !isfile(path)
        return false
    end
    lines = readlines(path)
    k = min(length(lines), n)
    return any(occursin(pattern, lines[i]) for i in 1:k)
end

function check_problem_files_and_rules(repo_root::String, problem::String)
    adnlp_path = joinpath(repo_root, "src", "ADNLPProblems", "$(problem).jl")
    jump_path = joinpath(repo_root, "src", "PureJuMP", "$(problem).jl")
    meta_path = joinpath(repo_root, "src", "Meta", "$(problem).jl")

    file_ok = Dict(
        "ADNLP" => isfile(adnlp_path),
        "PureJuMP" => isfile(jump_path),
        "Meta" => isfile(meta_path),
    )

    header_ok = first_lines_have_pattern(adnlp_path, "Hock and Schittkowski")

    adnlp_text = file_ok["ADNLP"] ? read(adnlp_path, String) : ""
    bounds_ok = occursin("lvar", adnlp_text) && occursin("uvar", adnlp_text) && occursin("x0", adnlp_text)

    meta_text = file_ok["Meta"] ? read(meta_path, String) : ""
    metadata_ok = occursin(":name => \"$(problem)\"", meta_text) &&
                  occursin(":best_known_upper_bound", meta_text)

    return Dict(
        "problem" => problem,
        "files" => file_ok,
        "header" => header_ok,
        "bounds" => bounds_ok,
        "metadata" => metadata_ok,
        "all_core_checks_pass" => all(values(file_ok)) && header_ok && bounds_ok && metadata_ok,
    )
end

checks = check_problem_files_and_rules(repo_root, "hs85")
println(JSON.json(checks, 2))

In [ ]:
function parse_numeric_vector_literal(text::String, name::String)
    m = match(Regex(name * raw"\s*=\s*T\[(.*?)\]", "s"), text)
    m === nothing && error("Could not locate vector literal for $(name)")
    body = m.captures[1]
    tokens = split(replace(body, '\n' => ' '), ',')
    vals = Float64[]
    for t in tokens
        s = strip(t)
        isempty(s) && continue
        push!(vals, parse(Float64, s))
    end
    return vals
end

adnlp_text = read(joinpath(repo_root, "src", "ADNLPProblems", "hs85.jl"), String)
code_lvar = parse_numeric_vector_literal(adnlp_text, "lvar")
code_uvar = parse_numeric_vector_literal(adnlp_text, "uvar")
code_x0 = parse_numeric_vector_literal(adnlp_text, "x0")

json_lvar = Float64.(extract["bounds"]["lvar"])
json_uvar = Float64.(extract["bounds"]["uvar"])
json_x0 = Float64.(extract["bounds"]["x0"])

println("Bounds and x0 consistency with extraction JSON:")
println("lvar match: ", code_lvar == json_lvar)
println("uvar match: ", code_uvar == json_uvar)
println("x0 match:   ", code_x0 == json_x0)

In [ ]:
try
    nlp = OptimizationProblems.ADNLPProblems.hs85()
    x0 = nlp.meta.x0
    f0 = NLPModels.obj(nlp, x0)
    c0 = NLPModels.cons(nlp, x0)

    println("ADNLP model summary:")
    println("name: ", nlp.meta.name)
    println("nvar: ", nlp.meta.nvar)
    println("ncon: ", nlp.meta.ncon)
    println("objective at x0: ", f0)
    println("min constraint value at x0 (for c(x) >= 0 style constraints): ", minimum(c0))
catch err
    println("Skipped ADNLP instantiation step. Reason: ", err)
end

In [ ]:
try
    using JuMP, Ipopt
    model = OptimizationProblems.PureJuMP.hs85_jump()
    optimize!(model)
    println("Ipopt termination status: ", termination_status(model))
    println("Objective value: ", objective_value(model))
catch err
    println("Skipped solve step. Reason: ", err)
    println("This is expected if Ipopt/JuMP are not available in the current notebook environment.")
end

## Manual vs AI Intervention Notes

What this notebook automates well:
- Structural checks (file existence, headers, basic metadata patterns).
- Consistency checks between extraction JSON and implementation bounds/x0.
- Basic model instantiation and optional solve sanity-check.

What still needs manual review:
- Ambiguous PDF interpretation and formula disambiguation.
- Naming choices and duplicate detection decisions.
- Semantic fidelity of objective/constraints against original publication.

This balance matches PR #412 workflow: AI accelerates drafting/checking, while humans validate mathematical intent and project policy decisions.